# 📊 Stage 4 - Publication-Quality Plots

Generates all final figures for **every condition** present in `{SAMPLE_ID}/Results/`.
Additionally produces a **multi-condition Brouwer diagram** aggregating all atmospheric
conditions of the sample.

---
**Figures produced per condition** (saved as PNG + PDF):

| Figure | File | Source |
|--------|------|---------|
| DRT stacked | `DRT_{condition}_Stacked.{png,pdf}` | stage3_drt.xlsx → DRT_Spectra |
| Nyquist overlay | `Nyquist_{condition}.{png,pdf}` | ISM files + stage3_fit.xlsx |
| Bode plot | `Bode_{condition}.{png,pdf}` | ISM files + stage3_fit.xlsx |
| Arrhenius 2×2 | `Arrhenius_{condition}.{png,pdf}` | stage3_fit.xlsx |
| τ consistency | `TauConsistency_{condition}.{png,pdf}` | stage3_fit.xlsx |

**Figures produced per sample** (in `Results/pO2/`):

| Figure | File |
|--------|----- |
| Brouwer p(O₂) | `Brouwer_Peak1_{SAMPLE_ID}.{png,pdf}` |

---
**Input**  : `{SAMPLE_ID}/Results/{condition}/stage3_drt.xlsx`
             `{SAMPLE_ID}/Results/{condition}/stage3_fit.xlsx`
             `{SAMPLE_ID}/ISM validation/{condition}/*.ism`

**Output** : PNG + PDF in `{SAMPLE_ID}/Results/{condition}/` sub-folders

> **Workflow**: Config → run all cells → inspect figures inline → PNG/PDF are saved automatically

## Quick links
- [Sample, geometry, FOCUS_T](#-stage-4--publication-quality-plots) — `SAMPLE_ID`, `CONDITION_FILTER`, `L_m`, `D_m`
- [PLOT_WINDOWS — per-(condition, T) axis crop](#-stage-4--publication-quality-plots) — `z_max`, `freq_lim`, `focus_peaks`
- [DRT stacked range](#-stage-4--publication-quality-plots) — `DRT_TAU_MAX`
- [Brouwer p(O₂) settings](#-step-2--brouwer-po-all-conditions) — `BROUWER_PEAK_ID`, `BROUWER_TEMPS` (also a live peak selector in Step 2b)
- [τ Arrhenius threshold](#-stage-4--publication-quality-plots) — `TAU_R2_THRESHOLD`

**Workflow:** edit the config cell → run the plot cell → use the Step 1b crop panel (↻ Replot Nyquist/Bode) and the Step 2b Brouwer peak selector without scrolling → Export PLOT_WINDOWS when satisfied.

In [ ]:
# =============================================================
#  CONFIGURATION — edit only this cell
# =============================================================

SAMPLE_ID = "SAMPLE_ID"   # sample folder inside EIS program/

# Optional: process only specific conditions (leave empty [] to process ALL)
CONDITION_FILTER = []  # e.g. ["SAMPLE_ID_Ar-SCCM_O2-SCCM_Tmax_Tmin_delta"]

# Sample geometry (same values as Stage 3)
L_m = None    # thickness [m]
D_m = None   # diameter [m]

# DRT stacked plot: x-axis upper limit [s]
# Increase if low-frequency electrode arc is important
DRT_TAU_MAX = 0.1

# Brouwer diagram: which peak_id to use (1 = highest-frequency process)
BROUWER_PEAK_ID = 1

# Temperatures to show in the Brouwer diagram (None = all)
BROUWER_TEMPS = None  # e.g. [400, 450, 500, 550, 600]

# τ Arrhenius consistency: R² threshold to flag a peak as physically real
TAU_R2_THRESHOLD = 0.97

# =============================================================
#  PLOT_WINDOWS — per-(condition, T) axis crop for saved figures
# =============================================================
# Visualisation only. The fit is always done on the full KK-validated range
# (Stage 3, unchanged). PLOT_WINDOWS only changes which portion is shown.
#
# Schema:
#   PLOT_WINDOWS[condition][T_int] = {
#       "z_max":     kOhm,    # Nyquist Z' upper limit (Z'' upper = same)
#       "z_min":     kOhm,    # Nyquist Z' lower limit (default 0)
#       "freq_min":  Hz,      # Bode lower freq
#       "freq_max":  Hz,      # Bode upper freq
#       "focus_peaks": [1, 2] # informational — which peaks the crop highlights
#   }
#
# Missing keys → auto-scale from data (current behaviour preserved).
# Whole dict empty → all figures full-range, byte-identical to pre-PLOT_WINDOWS output.
PLOT_WINDOWS: dict[str, dict[int, dict]] = {
    # Example:
    # "SAMPLE_ID_Ar-SCCM_O2-SCCM_Tmax_Tmin_delta": {
    #     600: {"z_max": 50, "freq_min": 100, "freq_max": 1e6, "focus_peaks": [1]},
    #     400: {"z_max": 500, "z_min": 0, "freq_min": 30, "freq_max": 1e5, "focus_peaks": [1, 2]},
    # },
}


def _resolve_window(condition: str, T_int: int) -> dict:
    """Return the PLOT_WINDOWS entry for (condition, T) or empty dict."""
    return PLOT_WINDOWS.get(condition, {}).get(int(T_int), {})


def _nyquist_xylim(window: dict) -> tuple:
    """Convert a PLOT_WINDOWS dict into (xlim, ylim) tuples (kOhm) or (None, None)."""
    xlim = ylim = None
    z_min = window.get("z_min", 0)
    if "z_max" in window:
        xlim = (z_min, window["z_max"])
        ylim = (0, window["z_max"])
    return xlim, ylim


def _bode_freqlim(window: dict) -> tuple | None:
    """Return (freq_min, freq_max) or None."""
    fmin = window.get("freq_min")
    fmax = window.get("freq_max")
    if fmin is None and fmax is None:
        return None
    return (fmin if fmin is not None else 1e-3, fmax if fmax is not None else 1e9)

In [ ]:
# Always render figures inline. Interactive exploration is provided by the
# ipywidgets control panels below (replot with crop windows / peak selection),
# which are more reliable here than the ipympl zoom/pan backend.
get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore[name-defined]

import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.ingest import load_ism
from pipeline.drt import clip_spectrum
from pipeline.plots import (
    apply_pub_style,
    plot_drt_stacked,
    plot_nyquist_multipanel,
    plot_bode,
    plot_arrhenius_panel,
    plot_tau_arrhenius_consistency,
    plot_tau_tracks,
    plot_brouwer,
    build_arrhenius_results,
    plot_ceff_magnitude,
)

# Apply publication style once
apply_pub_style()

SAMPLE_DIR   = NOTEBOOK_DIR / SAMPLE_ID
RESULTS_BASE = SAMPLE_DIR / "Results"

# Collect conditions that have completed Stage 3
all_conditions = sorted([
    d.name for d in RESULTS_BASE.iterdir()
    if d.is_dir()
    and (d / "stage3_fit.xlsx").exists()
    and (d / "stage3_drt.xlsx").exists()
])

conditions = (
    [c for c in all_conditions if c in CONDITION_FILTER]
    if CONDITION_FILTER else all_conditions
)

print(f"Sample     : {SAMPLE_ID}")
print(f"Geometry   : L = {L_m*1e3:.3f} mm  D = {D_m*1e3:.3f} mm")
print(f"Conditions with Stage 3 output ({len(conditions)}):")
for c in conditions:
    print(f"  {c}")


def _condition_window(condition: str) -> dict:
    """Condition-level PLOT_WINDOWS entry: only top-level string keys (skip per-T int keys)."""
    cw = PLOT_WINDOWS.get(condition, {})
    return {k: v for k, v in cw.items() if isinstance(k, str)}

## ▶ Step 1 — Figures per condition

For each condition: loads the validated EIS spectra and the Zarc parameters from Stage 3,
then produces DRT stacked, Nyquist, Bode, Arrhenius and τ consistency plots.

All figures are saved automatically to `Results/{condition}/` as PNG + PDF.

In [ ]:
# Collect all peak data across conditions (used later for the Brouwer diagram)
all_peaks_df_list = []

# Cache of in-memory data per condition — used by the live control panel below
_plot_cache: dict[str, dict] = {}

for condition in conditions:
    print(f"\n{'='*70}")
    print(f"Condition: {condition}")
    print(f"{'='*70}")

    res_dir  = RESULTS_BASE / condition
    val_dir  = SAMPLE_DIR / "ISM validation" / condition

    # ------------------------------------------------------------------
    # Load Stage 3 outputs
    # ------------------------------------------------------------------
    df_fit_peaks   = pd.read_excel(res_dir / "stage3_fit.xlsx",  sheet_name="Peaks")
    df_fit_summary = pd.read_excel(res_dir / "stage3_fit.xlsx",  sheet_name="Summary")
    df_drt_spectra = pd.read_excel(res_dir / "stage3_drt.xlsx",  sheet_name="DRT_Spectra")
    df_kk_sel      = pd.read_excel(res_dir / "stage2_kk.xlsx",   sheet_name="Selected")

    # Collect for Brouwer aggregation
    all_peaks_df_list.append(df_fit_peaks)

    temps_available = sorted(df_fit_summary["T_nominal"].unique())
    print(f"  Temperatures: {[int(t) for t in temps_available]}")

    # ------------------------------------------------------------------
    # Load ISM data (validated, same files used in Stage 3)
    # ------------------------------------------------------------------
    records    = {}   # {T_nominal: (freq, Z_re, Z_im)}
    fit_params = {}   # {T_nominal: {R0, R, tau, alpha}}

    for _, row in df_kk_sel.iterrows():
        T_nom  = int(row["T_nominal"])
        fname  = row["file"]
        f_min  = row["f_min_cut"] if pd.notna(row.get("f_min_cut")) else None
        f_max  = row["f_max_cut"] if pd.notna(row.get("f_max_cut")) else None

        ism_path = val_dir / fname
        if not ism_path.exists():
            print(f"  [WARN] Not found: {fname} — skipping T={T_nom}")
            continue

        rec  = load_ism(ism_path)
        freq, Z_re, Z_im = clip_spectrum(
            rec.freq, rec.Z_re, rec.Z_im, f_min, f_max
        )
        records[T_nom] = (freq, Z_re, Z_im)

        # Zarc parameters for this temperature
        sub = df_fit_peaks[df_fit_peaks["T_nominal"] == T_nom]
        if not sub.empty:
            sum_row = df_fit_summary[df_fit_summary["T_nominal"] == T_nom]
            R0_val  = float(sum_row["R0"].iloc[0]) if not sum_row.empty else None
            fit_params[T_nom] = {
                "R0":    R0_val,
                "R":     sub["R_i"].values.tolist(),
                "tau":   sub["tau_i"].values.tolist(),
                "alpha": sub["alpha_i"].values.tolist(),
            }

    # Cache for the interactive control panel (Step 3 below)
    _plot_cache[condition] = {
        "records":        records,
        "fit_params":     fit_params,
        "df_drt_spectra": df_drt_spectra,
        "df_fit_peaks":   df_fit_peaks,
        "df_fit_summary": df_fit_summary,
        "res_dir":        res_dir,
    }

    # ------------------------------------------------------------------
    # Output sub-directories
    # ------------------------------------------------------------------
    drt_dir  = res_dir / "DRT"
    nq_dir   = res_dir / "Nyquist-Bode"
    arr_dir  = res_dir / "Arrhenius"

    # Resolve condition-level plot window (None = full range)
    cw = _condition_window(condition)
    nyq_xlim, nyq_ylim = _nyquist_xylim(cw)
    bode_freq          = _bode_freqlim(cw)

    # ------------------------------------------------------------------
    # 1. DRT stacked plot
    # ------------------------------------------------------------------
    print("  → DRT stacked plot")
    if not df_drt_spectra.empty:
        fig_drt = plot_drt_stacked(
            df_spectra   = df_drt_spectra,
            condition    = condition,
            save_dir     = drt_dir,
            tau_max      = DRT_TAU_MAX,
        )
        plt.show()
        plt.close(fig_drt)
    else:
        print("    [SKIP] No DRT spectra data.")

    # ------------------------------------------------------------------
    # 2. Nyquist overlay (with optional condition-level crop)
    # ------------------------------------------------------------------
    print("  → Nyquist overlay" + (f"  [window z_max={cw.get('z_max')} kΩ]" if "z_max" in cw else ""))
    if records:
        fig_nq = plot_nyquist_multipanel(
            records    = records,
            fit_params = fit_params,
            condition  = condition,
            save_dir   = nq_dir,
            xlim       = nyq_xlim,
            ylim       = nyq_ylim,
        )
        plt.show()
        plt.close(fig_nq)

    # ------------------------------------------------------------------
    # 3. Bode plot (with optional condition-level freq window)
    # ------------------------------------------------------------------
    print("  → Bode plot" + (f"  [freq window {bode_freq}]" if bode_freq else ""))
    if records:
        fig_bode = plot_bode(
            records    = records,
            fit_params = fit_params,
            condition  = condition,
            save_dir   = nq_dir,
            freq_lim   = bode_freq,
        )
        plt.show()
        plt.close(fig_bode)

    # ------------------------------------------------------------------
    # 4. Arrhenius 2×2 panel
    # ------------------------------------------------------------------
    print("  → Arrhenius panel")
    if not df_fit_peaks.empty:
        fig_arr, results_all = plot_arrhenius_panel(
            df_peaks  = df_fit_peaks,
            L_m       = L_m,
            D_m       = D_m,
            condition = condition,
            save_dir  = arr_dir,
        )
        plt.show()
        plt.close(fig_arr)

        # Print activation energy summary table
        print(f"\n  Activation energies — {condition}")
        print(f"  {'Peak':<10} {'Ea_cond (eV)':<18} {'Ea_pol (eV)':<18} "
              f"{'Ea_C (eV)':<18} {'R2_cond':<10} {'R2_pol':<10} {'R2_C':<10}")
        print(f"  {'-'*94}")
        for r in results_all:
            def _fmt(v, e):
                return f"{v:.3f}±{e:.3f}" if not (np.isnan(v) or np.isnan(e)) else "N/A"
            def _r2(v):
                return f"{v:.4f}" if not np.isnan(v) else "N/A"
            print(f"  {r['Peak']:<10} {_fmt(r['Ea_cond'], r['Ea_cond_err']):<18} "
                  f"{_fmt(r['Ea_pol'], r['Ea_pol_err']):<18} "
                  f"{_fmt(r['Ea_C'], r['Ea_C_err']):<18} "
                  f"{_r2(r['R2_cond']):<10} {_r2(r['R2_pol']):<10} {_r2(r['R2_C']):<10}")
        print()

        # 4b. Effective-capacitance magnitude (read the process from C_eff, no labels)
        print("  → C_eff magnitude plot")
        fig_cap = plot_ceff_magnitude(
            df_peaks  = df_fit_peaks,
            condition = condition,
            save_dir  = arr_dir,
        )
        plt.show()
        plt.close(fig_cap)

    # ------------------------------------------------------------------
    # 5. τ Arrhenius consistency
    # ------------------------------------------------------------------
    print("  → τ consistency plot")
    if not df_fit_peaks.empty:
        fig_tau = plot_tau_arrhenius_consistency(
            df_peaks      = df_fit_peaks,
            condition     = condition,
            save_dir      = arr_dir,
            r2_threshold  = TAU_R2_THRESHOLD,
        )
        plt.show()
        plt.close(fig_tau)

        # 5b. tau-track diagnostic (cross-temperature peak alignment)
        print("  -> tau-track diagnostic")
        fig_trk = plot_tau_tracks(
            df_peaks  = df_fit_peaks,
            condition = condition,
            save_dir  = arr_dir,
        )
        plt.show()
        plt.close(fig_trk)

    print(f"  Figures saved in {res_dir.relative_to(NOTEBOOK_DIR)}")

print(f"\n{'='*70}")
print(f"Per-condition figures complete — {len(conditions)} condition(s).")

## ▶ Step 1b — Nyquist/Bode crop panel (no-scroll iteration)

Use this panel to **crop and re-save** a single Nyquist/Bode figure without re-running Step 1.

1. **Cond / T [°C]** — pick the spectrum to inspect.
2. **Z_min / Z_max kΩ** — Nyquist axis crop (`Z_max = 0` → auto-scale).
3. **f_min / f_max Hz** — Bode frequency window.
4. **💾 Save crop** — when checked, the cropped PNG/PDF overwrites the per-T file; leave unchecked for a preview only.
5. **↻ Replot Nyquist/Bode** — regenerate the figure in place from cached Step-1 data (no re-fit, no re-read).
6. **📤 Export PLOT_WINDOWS** — write a `plot_windows.py` snippet next to the sample folder so the crops persist across runs.

If `ipywidgets` is not installed, the cell prints a hint and exits cleanly.

In [ ]:
# Live control panel — edit window + replot from in-memory data (no recompute).
try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear
    _HAS_WIDGETS = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); control panel disabled. "
          "Install with: pip install ipywidgets")
    _HAS_WIDGETS = False


def _render_one(condition: str, T_int: int, save: bool = True) -> None:
    """Render Nyquist + Bode for a single (condition, T) using cached data.

    Reads ``PLOT_WINDOWS[condition][T_int]`` for crop limits. When ``save=True``,
    writes the cropped publication PNG/PDF; otherwise only displays inline.
    """
    cache = _plot_cache.get(condition)
    if cache is None:
        print(f"[WARN] no cached data for {condition} — run Step 1 first.")
        return
    if T_int not in cache["records"]:
        print(f"[WARN] T={T_int}°C not in cached records for {condition}.")
        return

    window = _resolve_window(condition, T_int)
    xlim, ylim = _nyquist_xylim(window)
    freq_lim   = _bode_freqlim(window)

    nq_dir = cache["res_dir"] / "Nyquist-Bode"

    # Single-T view: subset the records dict to just this T (cleaner visual)
    one_rec = {T_int: cache["records"][T_int]}
    one_fit = {T_int: cache["fit_params"].get(T_int)}

    fig_nq = plot_nyquist_multipanel(
        records=one_rec, fit_params=one_fit,
        condition=f"{condition}_T{T_int}",
        save_dir=nq_dir, xlim=xlim, ylim=ylim, save=save,
    )
    plt.show()
    fig_bd = plot_bode(
        records=one_rec, fit_params=one_fit,
        condition=f"{condition}_T{T_int}",
        save_dir=nq_dir, freq_lim=freq_lim, save=save,
    )
    plt.show()
    print(f"  window applied: xlim={xlim} kΩ, ylim={ylim} kΩ, freq={freq_lim} Hz"
          f"  (save={'PNG+PDF' if save else 'preview only'})")


def _set_window(condition: str, T_int: int, **kw) -> None:
    """Persist a per-(condition, T) entry in the in-memory PLOT_WINDOWS dict."""
    PLOT_WINDOWS.setdefault(condition, {}).setdefault(int(T_int), {})
    PLOT_WINDOWS[condition][int(T_int)].update(
        {k: v for k, v in kw.items() if v is not None}
    )


def _export_plot_windows(path: Path | None = None) -> Path:
    """Dump the in-memory PLOT_WINDOWS dict to a Python file for permanent reuse."""
    out = path or (SAMPLE_DIR / "Results" / "plot_windows.py")
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", encoding="utf-8") as fh:
        fh.write("# Auto-exported by 04_plots.ipynb control panel.\n")
        fh.write("# Paste this dict into PLOT_WINDOWS in cell-1 to make crops permanent.\n\n")
        fh.write("PLOT_WINDOWS = ")
        import pprint as _pp
        fh.write(_pp.pformat(PLOT_WINDOWS, width=110, sort_dicts=False))
        fh.write("\n")
    return out


if _HAS_WIDGETS and conditions:
    _cond0 = conditions[0]
    _Ts0   = sorted(_plot_cache.get(_cond0, {}).get("records", {}).keys(), reverse=True)
    _T0    = _Ts0[0] if _Ts0 else 600

    w_cond = W.Dropdown(options=conditions, value=_cond0, description="Cond:",
                        layout=W.Layout(width="420px"))
    w_T    = W.Dropdown(options=_Ts0, value=_T0, description="T [°C]:",
                        layout=W.Layout(width="200px"))
    w_zmin = W.FloatText(value=0, description="Z_min kΩ", layout=W.Layout(width="180px"))
    w_zmax = W.FloatText(value=0, description="Z_max kΩ",
                         layout=W.Layout(width="180px"),
                         tooltip="0 = auto-scale")
    w_fmin = W.FloatLogSlider(value=1, base=10, min=-2, max=8, step=0.1,
                              description="f_min Hz", readout_format=".2e",
                              layout=W.Layout(width="360px"))
    w_fmax = W.FloatLogSlider(value=1e6, base=10, min=-2, max=8, step=0.1,
                              description="f_max Hz", readout_format=".2e",
                              layout=W.Layout(width="360px"))
    w_save = W.Checkbox(value=False, description="💾 Save crop (PNG/PDF)",
                        tooltip="When True, the cropped figure also overwrites the per-T file.")
    w_go   = W.Button(description="↻ Replot Nyquist/Bode", button_style="primary",
                      layout=W.Layout(width="220px"))
    w_exp  = W.Button(description="📤 Export PLOT_WINDOWS",
                      button_style="success", layout=W.Layout(width="240px"))
    out    = W.Output()

    def _refresh_T(*_):
        ts = sorted(_plot_cache.get(w_cond.value, {}).get("records", {}).keys(), reverse=True)
        w_T.options = ts
        if ts and w_T.value not in ts:
            w_T.value = ts[0]
    w_cond.observe(_refresh_T, names="value")

    def _on_replot(_btn):
        # Persist user-set window into PLOT_WINDOWS, then replot.
        kw = {}
        if w_zmin.value:               kw["z_min"]    = float(w_zmin.value)
        if w_zmax.value:               kw["z_max"]    = float(w_zmax.value)
        if w_fmin.value > 0:           kw["freq_min"] = float(w_fmin.value)
        if w_fmax.value > 0:           kw["freq_max"] = float(w_fmax.value)
        if kw:
            _set_window(w_cond.value, int(w_T.value), **kw)
        with out:
            _clear(wait=True)
            _render_one(w_cond.value, int(w_T.value), save=bool(w_save.value))
    w_go.on_click(_on_replot)

    def _on_export(_btn):
        path = _export_plot_windows()
        with out:
            _clear(wait=True)
            print(f"PLOT_WINDOWS exported → {path}")
            print("Paste the dict into the PLOT_WINDOWS variable in cell-1 to persist.")
    w_exp.on_click(_on_export)

    _display(
        W.VBox([
            W.HBox([w_cond, w_T]),
            W.HBox([w_zmin, w_zmax]),
            W.HBox([w_fmin, w_fmax]),
            W.HBox([w_save, w_go, w_exp]),
            out,
        ])
    )
elif not conditions:
    print("[INFO] No conditions available — run Step 1 first.")

## ▶ Step 2 — Brouwer p(O₂) — all conditions

Aggregates data for peak `BROUWER_PEAK_ID` from **all** conditions and plots log₁₀(σ) vs log₁₀(p(O₂)).

Each symbol encodes a temperature (400–600 °C). Slope guides −¼, plateau, +¼ for interpretation.

> **Note**: the Brouwer diagram is physically meaningful only if the same `peak_id` represents the same process across all conditions. Verify using the C_eff magnitude and Arrhenius behaviour.

In [ ]:
if all_peaks_df_list:
    df_all_peaks = pd.concat(all_peaks_df_list, ignore_index=True)

    # Check that Peak 1 is present for multiple conditions
    peak1_data = df_all_peaks[df_all_peaks["peak_id"] == BROUWER_PEAK_ID]
    n_cond_p1  = peak1_data["condition"].nunique() if not peak1_data.empty else 0
    print(f"Peak {BROUWER_PEAK_ID} data found in {n_cond_p1} condition(s) of {len(conditions)} total.")

    if n_cond_p1 < 2:
        print("  [SKIP] Brouwer diagram requires ≥ 2 conditions. "
              "Run Stage 3 on more atmospheric conditions first.")
    else:
        brouwer_dir = RESULTS_BASE / "pO2"
        fig_brouwer = plot_brouwer(
            df_all       = df_all_peaks,
            save_dir     = brouwer_dir,
            sample_name  = SAMPLE_ID,
            peak_id      = BROUWER_PEAK_ID,
            temps_to_plot= BROUWER_TEMPS,
            add_slopes   = True,
        )
        plt.show()
        print(f"Brouwer diagram saved in {brouwer_dir.relative_to(NOTEBOOK_DIR)}")

        # Print pO2 data table for verification
        print(f"\nPeak {BROUWER_PEAK_ID} data used for Brouwer diagram:")
        tbl = (
            peak1_data[["condition", "T_nominal", "pO2_mean", "R_i", "sigma_Sm_i"]]
            .sort_values(["T_nominal", "pO2_mean"])
            .reset_index(drop=True)
        )
        tbl["lg_pO2"]   = tbl["pO2_mean"].apply(lambda x: f"{np.log10(x):.3f}")
        tbl["lg_sigma"] = tbl["sigma_Sm_i"].apply(
            lambda x: f"{np.log10(x/100):.3f}" if x > 0 else "N/A")
        display(tbl[["condition", "T_nominal", "lg_pO2", "lg_sigma", "R_i"]])
else:
    print("No conditions processed — nothing to aggregate.")

## ▶ Step 2b — Brouwer: select peak and temperatures

Select a **peak** and (optionally) a **temperature** subset, then press
**↻ Replot Brouwer**. The figure `Brouwer_Peak{N}_{sample}.{png,pdf}` is saved to `Results/pO2/`.
Requires Step 2 to have been run first (uses `df_all_peaks` in memory).

In [ ]:
# Brouwer selector — replot the p(O2) diagram for a chosen peak / temperatures / conditions.
# Uses df_all_peaks built in Step 2 (in memory); never recomputes fits.
try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear
    _HAS_WIDGETS_BR = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); Brouwer panel disabled.")
    _HAS_WIDGETS_BR = False

if _HAS_WIDGETS_BR and ("df_all_peaks" in dir()) and not df_all_peaks.empty:
    _brouwer_dir = RESULTS_BASE / "pO2"
    _peak_ids  = sorted(int(p) for p in df_all_peaks["peak_id"].unique())
    _all_temps = sorted(int(t) for t in df_all_peaks["T_nominal"].unique())
    _all_conds = sorted(df_all_peaks["condition"].unique())

    w_peak = W.Dropdown(options=_peak_ids,
                        value=BROUWER_PEAK_ID if BROUWER_PEAK_ID in _peak_ids else _peak_ids[0],
                        description="Peak:", layout=W.Layout(width="180px"))
    w_temps = W.SelectMultiple(options=_all_temps, value=tuple(_all_temps),
                               description="T [°C]:", rows=min(9, len(_all_temps)),
                               layout=W.Layout(width="180px"))
    # Condition selector — deselect e.g. the Ar condition to drop it from the diagram.
    w_conds = W.SelectMultiple(options=_all_conds, value=tuple(_all_conds),
                               description="Cond:", rows=min(6, len(_all_conds)),
                               layout=W.Layout(width="440px"))
    w_slopes = W.Checkbox(value=True, description="Slope guides (-1/4, plateau, +1/4)")
    w_br_go = W.Button(description="↻ Replot Brouwer", button_style="primary",
                       layout=W.Layout(width="200px"))
    out_br = W.Output()

    def _on_brouwer(_btn):
        peak_id = int(w_peak.value)
        sel_T   = [int(t) for t in w_temps.value]
        temps   = sel_T if (sel_T and len(sel_T) != len(_all_temps)) else None
        sel_C   = list(w_conds.value) or _all_conds
        df_sel  = df_all_peaks[df_all_peaks["condition"].isin(sel_C)]
        sub     = df_sel[df_sel["peak_id"] == peak_id]
        n_cond  = sub["condition"].nunique() if not sub.empty else 0
        with out_br:
            _clear(wait=True)
            if n_cond < 2:
                print(f"[SKIP] Peak {peak_id} present in only {n_cond} selected condition(s); "
                      "Brouwer needs >= 2. Select more conditions.")
                return
            fig = plot_brouwer(
                df_all=df_sel, save_dir=_brouwer_dir,
                sample_name=SAMPLE_ID, peak_id=peak_id,
                temps_to_plot=temps, add_slopes=bool(w_slopes.value),
            )
            plt.show()
            print(f"Saved -> {(_brouwer_dir / f'Brouwer_Peak{peak_id}_{SAMPLE_ID}.png')}")
            print(f"  peak={peak_id}  temps={'all' if temps is None else temps}")
            print(f"  conditions ({n_cond}): {sel_C}")
    w_br_go.on_click(_on_brouwer)

    _display(W.VBox([W.HBox([w_peak, w_temps, w_conds, w_slopes]), w_br_go, out_br]))
elif _HAS_WIDGETS_BR:
    print("[INFO] df_all_peaks not available — run Step 2 first.")

## ▶ Output summary

All figures have been exported to the sub-folders of `Results/{condition}/`:
- `DRT/`           — DRT γ(τ) stacked per temperature
- `Nyquist-Bode/`  — Nyquist overlay + Bode
- `Arrhenius/`     — Arrhenius 2×2 panel + τ consistency
- `pO2/`           — Brouwer p(O₂) diagram (multi-condition)

Each figure is saved as **PNG** (quick preview) and **PDF** (publication ready).